In [ ]:
import healpy as hp
import numpy as np
import math

#Planck TT data
data_Planck = np.loadtxt('./experimental_data/COM_PowerSpect_CMB-TT-full_R3.01.txt')
ls_Planck_TT = data_Planck[:, 0]
dl_Planck_TT = data_Planck[:, 1]
sdl_Planck_TT1 = data_Planck[:, 2]
sdl_Planck_TT2 = data_Planck[:, 3]
round_ls_Pl_TT = np.round(ls_Planck_TT)

#Planck TE data
data_Planck = np.loadtxt('./experimental_data/COM_PowerSpect_CMB-TE-full_R3.01.txt')
ls_Planck_TE = data_Planck[:, 0]
dl_Planck_TE = data_Planck[:, 1]
sdl_Planck_TE1 = data_Planck[:, 2]
sdl_Planck_TE2 = data_Planck[:, 3]
round_ls_Pl_TE = np.round(ls_Planck_TE)

#Planck EE data
data_Planck = np.loadtxt('./experimental_data/COM_PowerSpect_CMB-EE-full_R3.01.txt')
ls_Planck_EE = data_Planck[:, 0]
dl_Planck_EE = data_Planck[:, 1]
sdl_Planck_EE1 = data_Planck[:, 2]
sdl_Planck_EE2 = data_Planck[:, 3]
round_ls_Pl_EE = np.round(ls_Planck_EE)

from CMBFeatureNet import generate_camb_power_spectra
Power_spectra = generate_camb_power_spectra(67.4, 0.02237, 0.1200, 0.06, 0, tau=0.0544,  
                    As=2.1e-9, ns=0.9649, halofit_version='mead', lmax=2507)

dlstt = Power_spectra.tt[:len(round_ls_Pl_TT)]
dlste = Power_spectra.te[:len(round_ls_Pl_TE)]
dlsee = Power_spectra.ee[:len(round_ls_Pl_EE)]

In [ ]:
Tcmb = 2.7255
CF = (Tcmb*10**6)**2

#Symmetrize errors:
def symm(splus,sminus):
    return (splus + sminus)/2 

#Converting the c_ls^TT to Dls
def Dls(l,CTT):
    Dl = [l[i]*(l[i]+1)*CTT[i]/(2*math.pi) for i in range(len(l))]
    return Dl

#Conerting Dls to the c_ls^TT
def Cls(l,DlTT):
    ClTT = [(2*math.pi)/(CF*l[i]*l[i]+1)*DlTT[i] for i in range(len(l))]
    return ClTT

# Extract the required spectra
cl_tt = Cls(round_ls_Pl_TT,dlstt)  # TT
cl_ee = Cls(round_ls_Pl_TT,dlste)  # EE
cl_bb = np.zeros_like(cl_ee)  # BB (set to zero if not considering B-modes)
cl_te = Cls(round_ls_Pl_EE,dlsee)  # TE

# Define the nside parameter (same as your TT map)
nside = 2048  # Adjust as needed

# Generate the Q and U maps (polarization maps)
maps = hp.synfast([cl_tt, cl_ee, cl_bb, cl_te], nside=nside, new=True, pol=True)

# Extract the polarization maps
T_map, Q_map, U_map = maps  # T, Q, and U components

# Save or plot the E-mode map
hp.mollview(Q_map, title="E-mode (Q component)", cmap="coolwarm")
hp.graticule()

In [ ]:
hp.mollview(U_map, title="E-mode (U component)", cmap="coolwarm")
hp.graticule()